# Run Your Code on a GPU

We're going to write a program that runs on a **GPU** — the same kind of chip
that runs ChatGPT — and watch it change a photo using **2.4 million threads at
once**.

### How this works

I'll build the code live and you follow along in your own copy. Two builds:

1. **Greyscale** — you type the whole program: every CUDA call, and the kernel.
2. **Blur** — a new file with the same `main()` already written; you type only
   the kernel, straight from the slide.

Then we'll race it against a CPU and see what that has to do with ChatGPT.

**If you fall behind, don't panic.** Under each build there's a
🛟 **finished version** cell — click it, copy the whole thing over your cell, and
you're caught up. Nothing later depends on you having typed it yourself.

---

### Two things before we start

1. **Turn the GPU on:** Runtime → Change runtime type → **T4 GPU** → Save.
2. **Run the grey cell below** (hover over it and press ▶, or Shift+Enter).

In [ ]:
#@title ▶ Run me first — sets everything up { display-mode: "form" }
import os, sys, subprocess, importlib

REPO = "/content/Workshop2"
URL = "https://github.com/daryl-888/Workshop2.git"

if os.path.isdir(os.path.join(REPO, ".git")):
    # A runtime that has been alive for a while still has an OLD copy of the
    # workshop. Bring it up to date. (Your own files in student/ are untouched.)
    subprocess.run(["git", "-C", REPO, "fetch", "--depth", "1", "--quiet", "origin", "main"])
    subprocess.run(["git", "-C", REPO, "reset", "--hard", "--quiet", "origin/main"])
elif os.path.isdir("lab") and os.path.isdir("images"):
    REPO = os.getcwd()                           # running locally, inside the repo
else:
    subprocess.run(["git", "clone", "--depth", "1", "--quiet", URL, REPO])

if not os.path.isdir(REPO):
    raise SystemExit("Couldn't download the workshop. Ask the facilitator to make "
                     "the repo PUBLIC (Settings -> General -> Change visibility).")

os.chdir(REPO)
if os.path.join(REPO, "lab") not in sys.path:
    sys.path.insert(0, os.path.join(REPO, "lab"))
import labkit as lab
lab = importlib.reload(lab)                      # in case an older copy was already loaded
lab.setup()

---
# 1 · Two computers in one box

Your machine has two very different processors in it, and they're good at
opposite things.

|  | **CPU** | **GPU** |
|---|---|---|
| how many workers | a handful | thousands |
| how fast is one | very | not very |
| best at | anything, one job after another | the *same* small job, a million times |
| memory | your RAM | **its own, separate RAM** |

That last row is the one that catches people out. **The GPU can't see your
data.** You have to hand it over, and ask for the answer back.

So every CUDA program does the same five things:

1. ask the GPU for some memory
2. copy your data into it
3. run your code there — thousands of copies at once
4. copy the answer back
5. give the memory back

A function that runs on the GPU is called a **kernel**. And the one idea behind
all of this:

> ### One thread per output element.
> One pixel, one thread. Two and a half million of them, at the same moment.

---
# 2 · Build 1: greyscale

The cell below is `student/main.cu`. Every time you change it: run the cell
(that saves the file), then run the check cell under it.

**`main()` starts empty and we type all of it.** Each piece has a STEP slide
with the exact lines on it. Four of those lines call small helpers from
`lab/gpulab.h` — `loadImage`, `makeImage`, `saveImage`, `freeImage` — because
reading a `.ppm` isn't CUDA. Everything else is.

| STEP | what you type | what it does |
|---|---|---|
| 1 | `loadImage()`, `makeImage(...)`, the two pointers, then `cudaMalloc(&in_d, img.bytes);` ×2 | the photo into CPU memory; ask the GPU for memory and write its address into `in_d` |
| 2 | `cudaMemcpy(in_d, img.data, img.bytes, cudaMemcpyHostToDevice);` | destination, source, how many, which way |
| 3 | `dim3 block(16, 16); dim3 grid(...); imageKernel<<<grid, block>>>(...); checkKernel(...)` | how many threads, run it, then wait and ask if it worked |
| 3 | the kernel body | what one thread does with its one pixel |
| 4 | `cudaMemcpy(out.data, out_d, img.bytes, cudaMemcpyDeviceToHost); saveImage(...)` | the same call, reversed; then write the file |
| 5 | `cudaFree` ×2, `freeImage` ×2, `return 0;` | give it all back — nothing does this for you |

Type them in order. Nothing runs until you run the cell, so a half-typed
`main()` is fine while we go.

In [ ]:
%%writefile student/main.cu
#include "lab/gpulab.h"

// ---------------------------------------------------------------
//  STEP 3, the kernel — runs once per thread, each on its own pixel.
// ---------------------------------------------------------------
__global__ void imageKernel(unsigned char* out, unsigned char* in, int w, int h) {

    /* YOUR CODE: STEP 3 — the kernel body, about 8 lines */

}

// ---------------------------------------------------------------
//  The host code — the five steps. All of it is typed, from the
//  STEP slides. (loadImage, makeImage, saveImage and freeImage are
//  small helpers that live in lab/gpulab.h.)
// ---------------------------------------------------------------
int main() {

    /* YOUR CODE: STEP 1 to STEP 5 — setup, malloc, copy over, launch, copy back, free */

}

Compile, run, check. `nvcc` splits the file in two: the kernel goes to the GPU,
everything else to the CPU. If something's wrong, the message below says *which
step* to look at.

In [ ]:
lab.run()

In [ ]:
#@title 🛟 Fell behind? The finished greyscale program { display-mode: "form" }
lab.solution("gray")

In [ ]:
lab.show()

### What just happened

Five CUDA calls and one kernel — that was the entire program. Two and a half
million threads each did about five instructions and stopped, and nobody wrote a
loop over the pixels.

That's the whole trick, and it's why the chip is built the way it is. If every
thread runs the same instruction, you don't need thousands of separate control
units. Strip those out, spend the space on arithmetic instead, and you get a
processor with thousands of tiny workers.

---
# 3 · Build 2: blur — new file, same `main()`

A blur replaces each pixel with the **average of the pixels around it** — for
radius 3, the 7×7 square centred on it.

This time the file is `student/blur.cu`, and **`main()` is already written** —
it is the same five steps you just typed, character for character, with the
kernel's name changed. Look at it and check. **The only thing you type in this
file is the kernel**: copy the whole function from the slide into the marked
spot.

One wrinkle inside it: a pixel in the corner has no neighbours above or to its
left. So the kernel checks each neighbour before using it, counts how many it
actually found, and divides by **that** — not by 49. Divide by 49 and the edges
come out dark.

In [ ]:
%%writefile student/blur.cu
#include "lab/gpulab.h"

#define BLUR_SIZE 3     // radius -> a 7x7 box around each pixel

// ---------------------------------------------------------------
//  THE KERNEL — copy the whole function from the slide here.
// ---------------------------------------------------------------

/* YOUR CODE: the blur kernel — __global__ void blurKernel(...) { ... } */

// ---------------------------------------------------------------
//  The host code — the same five steps as greyscale, already written.
//  Nothing to type below this line.
// ---------------------------------------------------------------
int main() {
    Image img = loadImage();
    Image out = makeImage(img.w, img.h);
    unsigned char *in_d, *out_d;

    // STEP 1
    cudaMalloc(&in_d,  img.bytes);
    cudaMalloc(&out_d, img.bytes);

    // STEP 2
    cudaMemcpy(in_d, img.data, img.bytes, cudaMemcpyHostToDevice);

    // STEP 3
    dim3 block(16, 16);
    dim3 grid((img.w + 15) / 16, (img.h + 15) / 16);
    blurKernel<<<grid, block>>>(out_d, in_d, img.w, img.h);
    checkKernel("blurKernel");

    // STEP 4
    cudaMemcpy(out.data, out_d, img.bytes, cudaMemcpyDeviceToHost);
    saveImage(out, "build/out.ppm");

    // STEP 5
    cudaFree(in_d);
    cudaFree(out_d);
    freeImage(img);
    freeImage(out);
    return 0;
}

In [ ]:
lab.run("blur")

In [ ]:
#@title 🛟 Fell behind? The finished blur program { display-mode: "form" }
lab.solution("blur")

In [ ]:
lab.show()

### 🎉 Same five steps, different picture

Look at the bottom row — the zoomed-in crop, where the softening is obvious. (On
the full photo it's real but easy to miss; a 7-pixel blur on a 1920-pixel-wide
picture gets subtle once it's shrunk to fit a screen.)

Scroll back up and compare the two `main()`s. They are the same program. The
host code is a shell; what the GPU actually *does* lives entirely in the kernel.

**Try this:** change `#define BLUR_SIZE 3` to `15`, re-run the cell and the
check. Each thread is now averaging 961 pixels instead of 49 — twenty times the
work — and it still finishes instantly. Try `31` if you like.

---
# 4 · How fast was that, really?

The same blur, done both ways: once as an ordinary loop on the CPU, once on the
GPU. Nothing to write — just run it and look at where the time goes.

In [ ]:
lab.run("race")

### Two numbers, and the gap between them

**The blur itself is hundreds of times faster than a CPU core.** Not because a
GPU core is fast — one GPU core is *slower* than a CPU core. Because there are
thousands of them, and this job splits into millions of identical independent
pieces.

**But counting the copying, it's much less impressive** — most of the GPU's time
went on moving bytes, not computing. That's the price of the GPU being a
separate machine with its own memory.

It's also the answer to something you may have wondered about: *why do people
care so much whether a model "fits" on a GPU?* Because the weights get loaded on
once and left there. Shipping them across for every single word would be
hopeless.

---
# 5 · What this has to do with ChatGPT

Follow the shape.

**Your blur.** Each output pixel is a sum over its neighbours. One thread per
pixel.

**Multiplying two matrices.** Each output number is a sum of products of a row
and a column. One thread per number. *The same shape of program* — the "which
one am I", the bounds check, the loop that adds things up. Only the arithmetic
in the middle is different.

**A language model.** Almost everything expensive inside one is a matrix
multiply: turning words into numbers, the attention step that works out which
words matter to each other, and the big layers in between. Producing **one word**
means billions of these little sums — each one an independent "one thread per
output number" job. Then it does it all again for the next word.

That's the real answer to *why GPUs and not CPUs*. Not that GPUs are
mysteriously fast — but that this job is millions of identical independent sums,
and that's the one thing a stadium of slow workers beats a handful of brilliant
ones at.

In [ ]:
lab.llm_math()

---
## What you did today

- Ran your own code on a genuinely different processor
- Handed data to a machine that couldn't otherwise see it, and got it back
- Pointed 2.4 million threads at 2.4 million pixels, exactly one each
- Made a real picture change, twice
- Measured it against a CPU, and found where the time actually goes

**If you want to keep going:** there are extra programs in `lab/extras/` — what
happens when you forget the copy back, what the bounds check is really
protecting you from, and a matrix multiply where changing *which* memory each
thread reads makes it several times faster for free.